![First-Time Environment Setup — get up and running locally: install dependencies, verify tools, prepare directories, run pipeline, verify artifacts](../docs/assets/ecg-first-time-environment-setup-banner.png)

# Step 0 — Environment setup, local-state preflight, and governed artifact generation — v2.45

This notebook prepares a fresh clone of `ecg_anomaly_detection` for the public notebook workflow.

It owns setup and local artifact generation so downstream notebooks can stay focused:

1. [`00-environment-setup-and-artifact-generation.ipynb`](./00-environment-setup-and-artifact-generation.ipynb) — this notebook
2. [`01-narrative-walkthrough.ipynb`](./01-narrative-walkthrough.ipynb)
3. [`02-high-performing-gradient-boosting-validation.ipynb`](./02-high-performing-gradient-boosting-validation.ipynb)

<div role="note" aria-label="Research and educational use boundary" style="border: 3px solid #007c91; border-left-width: 8px; border-radius: 6px; background: #eefbff; color: #102a43; padding: 1rem 1.25rem; margin: 1rem 0; line-height: 1.5;"><strong style="display: block; font-size: 1.15rem; margin-bottom: 0.35rem;">Research and educational use boundary</strong>This repository is a modernization, reproducibility, and data-engineering case study. It is not medical software, clinical software, diagnostic software, or production ML. Generated local validation evidence is not benchmark evidence.</div>


## Version history and change log

| Version | Change |
|---:|---|
| 2.45 | Added repository-relative workflow links and accessible, consistently styled callout panels for use boundaries, destructive cleanup, and blocked-state guidance. |
| 2.44 | Repaired strict readability patch placement for cell-magic code cells. `COMMENT MAP` blocks now remain present while preserving Jupyter’s requirement that cell magics such as `%%bash` stay on the first line. |
| 2.43 | Completed the strict notebook readability contract by adding missing `COMMENT MAP` blocks to every Step 0 code cell. This is a documentation/readability repair only; strict artifact-generation behavior from v2.42 is preserved. |
| 2.42 | Made environment bootstrap strict. Missing repository root, missing `uv`, failed `uv sync`, or failed CLI verification now stop Step 0 instead of being converted into a successful setup cell. |
| 2.41 | Made preflight strict as well as pipeline execution strict. Unsafe local state now stops Step 0 instead of allowing later cells to convert missing artifacts into downstream skip behavior. |
| 2.40 | Restored strict artifact-generation semantics for the public 00 → 01 → 02 workflow. Step 0 now fails fast when the governed pipeline fails or required downstream artifacts are missing, instead of converting PIPE-006 or other failures into a successful notebook run. |
| 2.32 | Replaced selector-based subprocess stream handling with simpler merged stdout/stderr line iteration. This removes the Pylance `readline` attribute warning from the public notebook while preserving progress output, local log capture, and controlled blocked-state reporting. |
| 2.31 | Changed known blocked states from notebook-terminating exceptions into controlled public UX output. Preflight, pipeline, and readiness cells now report blocked/incomplete states without traceback walls for expected local or PIPE-006 conditions. Expanded internal Python comments and helper docstrings so the notebook is easier to audit as a public example. |
| 2.30 | Reworked the governed pipeline execution cell from raw `%%bash` failure propagation into a Python-controlled subprocess runner. Known pipeline/config failures, especially PIPE-006 mixed channel identity, now write a local status file and print actionable diagnostics without dumping a `CalledProcessError` wall. Added helper-function docstrings, heavier inline Python comments, latest-run/deduplicated channel diagnostics, and safer readiness behavior. |
| 2.25 | Hardened Step 0 around real observed failure modes: hidden `.acquire-*` scratch directories, missing directory contracts, manifestless raw files, interrupted acquisition staging, stale generated outputs, and the confirmed PIPE-006 channel-identity contract failure. Reclassifies mixed `MLII` / `V5` window shards as a non-cleanup pipeline/config issue rather than a retry loop. Adds stronger diagnostics, safer cleanup posture, and clearer downstream readiness behavior. |
| 2.0 | Split setup and artifact generation into Step 0, with preflight checks, opt-in cleanup, retry logic, and downstream completion checks. |
| 1.0 | Initial Step 0 setup notebook created for the public notebook workflow. |


## 1. Start from a fresh clone

Recommended reviewer path:

1. Clone the repository.
2. Start JupyterLab from the repository root.
3. Run this Step 0 notebook from top to bottom.
4. Continue to the [`01` narrative walkthrough](./01-narrative-walkthrough.ipynb).
5. Run the public [`02` gradient boosting validation example](./02-high-performing-gradient-boosting-validation.ipynb).

The generated data and artifacts are intentionally ignored by Git. A fresh clone will not have them.


## 2. Local terminal launcher

Run this from a terminal at repository root if JupyterLab is not already open. The [environment reproducibility guide](../docs/environment-reproducibility.md) documents the supported notebook environment and kernel setup.

```bash
command -v uv >/dev/null 2>&1 || curl -LsSf https://astral.sh/uv/install.sh | sh
export PATH="$HOME/.local/bin:$PATH"
uv sync --group notebooks
uv run ecg-data --help >/dev/null
uv run --group notebooks jupyter lab notebooks/00-environment-setup-and-artifact-generation.ipynb
```

The cells below also include a guarded launcher template. It is disabled by default to avoid accidentally installing tools or launching nested JupyterLab sessions from inside an already-running notebook.


> **CODE CELL FUNCTION**
>
> Optional local launcher template. Disabled by default. Set `RUN_LOCAL_LAUNCHER=1` only when intentionally using this cell as a setup/launch helper.


In [ ]:
%%bash
# COMMENT MAP:
# - Establish repository-local paths used by later setup and validation cells.
# - Keep this cell side-effect-light: define constants and inspect state only.
# - Avoid hiding path assumptions so public reviewers can see where artifacts live.
# - Downstream cells should reuse these paths instead of reconstructing them ad hoc.
set -Eeuo pipefail

RUN_LOCAL_LAUNCHER=0
NOTEBOOK_PATH="notebooks/00-environment-setup-and-artifact-generation.ipynb"
started_at_epoch=$(date +%s)

elapsed() {
  local now_epoch
  now_epoch=$(date +%s)
  local elapsed_seconds=$((now_epoch - started_at_epoch))
  printf '%02dm %02ds' $((elapsed_seconds / 60)) $((elapsed_seconds % 60))
}

printf '[launcher] RUN_LOCAL_LAUNCHER=%s.\n' "$RUN_LOCAL_LAUNCHER"

if [[ "$RUN_LOCAL_LAUNCHER" != "1" ]]; then
  printf '[launcher] Disabled by default. Set RUN_LOCAL_LAUNCHER=1 to run this launcher intentionally.\n'
  printf '[launcher] This template would verify uv, sync notebook dependencies, validate the CLI, and launch JupyterLab.\n'
  exit 0
fi

printf '[launcher] Verifying repository root...\n'
test -f pyproject.toml || {
  printf '[launcher] ERROR: pyproject.toml not found. Start from repository root.\n' >&2
  exit 1
}

printf '[launcher] Checking uv availability...\n'
if ! command -v uv >/dev/null 2>&1; then
  printf '[launcher] uv not found. Installing uv with official installer...\n'
  curl -LsSf https://astral.sh/uv/install.sh | sh
  export PATH="$HOME/.local/bin:$PATH"
fi

printf '[launcher] uv version: %s\n' "$(uv --version)"
printf '[launcher] Syncing notebook dependency group. This can take several minutes on first run...\n'
uv sync --group notebooks
printf '[launcher] Dependency sync complete after %s.\n' "$(elapsed)"

printf '[launcher] Verifying project CLI...\n'
uv run ecg-data --help >/dev/null
printf '[launcher] CLI verification passed.\n'

printf '[launcher] Launching JupyterLab for %s...\n' "$NOTEBOOK_PATH"
uv run --group notebooks jupyter lab "$NOTEBOOK_PATH"


## 3. Environment bootstrap and runtime diagnostics

These cells verify that the notebook is running from repository root, that `uv` is available, and that the project CLI can be resolved.


> **CODE CELL FUNCTION**
>
> Sync the notebook dependency group and verify the project CLI.


In [ ]:
%%bash
# COMMENT MAP:
# - Verify the active Python/kernel environment before running pipeline work.
# - Surface dependency or kernel mismatches as real setup failures, not soft skips.
# - Keep diagnostics explicit so a reviewer knows whether to fix uv sync or kernel selection.
# - Do not mutate generated artifacts in this cell.
# CODE CELL FUNCTION:
# Bootstrap the notebook dependency group and verify the project CLI.
#
# This cell is strict. Environment bootstrap failures stop Step 0 because downstream notebooks cannot run without the uv-backed dependency group and project CLI.

set -u

printf '[bootstrap] Checking repository root...\n'
if [[ ! -f pyproject.toml ]]; then
  printf '[bootstrap] BLOCKED: pyproject.toml not found. Start JupyterLab from repository root.\n'
  exit 1
fi

printf '[bootstrap] Checking uv availability...\n'
if ! command -v uv >/dev/null 2>&1; then
  printf '[bootstrap] BLOCKED: uv is not installed or not on PATH.\n'
  printf '[bootstrap] Install uv from https://docs.astral.sh/uv/getting-started/installation/ and reopen this notebook.\n'
  exit 1
fi

printf '[bootstrap] uv found: %s\n' "$(uv --version)"
printf '[bootstrap] Syncing notebook dependency group...\n'
uv sync --group notebooks
sync_exit=$?

if [[ "$sync_exit" -ne 0 ]]; then
  printf '[bootstrap] BLOCKED: uv sync --group notebooks failed with exit code %s.\n' "$sync_exit"
  printf '[bootstrap] Review the uv output above, fix the environment issue, then rerun this cell.\n'
  exit 1
fi

printf '[bootstrap] Verifying project CLI...\n'
uv run ecg-data --help >/dev/null
cli_exit=$?

if [[ "$cli_exit" -ne 0 ]]; then
  printf '[bootstrap] BLOCKED: project CLI verification failed with exit code %s.\n' "$cli_exit"
  printf '[bootstrap] Re-run uv sync --group notebooks or inspect package installation.\n'
  exit 1
fi

printf '[bootstrap] Environment bootstrap complete.\n'


> **CODE CELL FUNCTION**
>
> Print runtime diagnostics without modifying local data or artifacts.


In [ ]:
# CODE CELL FUNCTION:
# COMMENT MAP:
# - Report kernel, Python, working-directory, and repository-root assumptions.
# - Keep the cell read-only; diagnostics must not mutate local data or artifacts.
# - Surface environment context before expensive acquisition or pipeline work starts.
# - Leave hard prerequisite enforcement to later strict preflight/pipeline cells.
# Display runtime diagnostics so reviewers know where the notebook is executing.
#
# This cell is intentionally read-only. It does not create, delete, or modify
# repository files. Its purpose is to make environment assumptions visible before
# any artifact-generation work begins.

from __future__ import annotations

import os
import platform
import sys
from pathlib import Path

# `Path.cwd()` is the kernel working directory. For this notebook, it should be
# the repository root. If it is not, later cells print clear blocked guidance
# instead of throwing a raw traceback.
working_directory = Path.cwd()

# These two paths are a lightweight repository-root check. We avoid importing the
# package here because a diagnostic cell should be cheap, side-effect-free, and
# safe to run before dependency bootstrap finishes.
repository_root_detected = Path("pyproject.toml").exists() and Path("configs").exists()

# These environment markers help distinguish common reviewer execution contexts.
# The notebook does not change behavior based on these alone; they are surfaced so
# the user can diagnose why local file paths or dependency installs may differ.
running_in_codespaces = bool(os.environ.get("CODESPACES"))
# A loaded google.colab module is Colab's own reliable self-identification signal.
running_in_colab_like_runtime = "google.colab" in sys.modules

print(f"[runtime] Python: {sys.version.split()[0]}")
print(f"[runtime] Platform: {platform.platform()}")
print(f"[runtime] Working directory: {working_directory}")
print(f"[runtime] Repository root detected: {repository_root_detected}")
print(f"[runtime] GitHub Codespaces: {running_in_codespaces}")
print(f"[runtime] Google Colab-style runtime: {running_in_colab_like_runtime}")

# Keep the final line explicit because this cell is often the first executable
# confirmation that the notebook is pointed at the right checkout.
if not repository_root_detected:
    print("[runtime] BLOCKED: reopen JupyterLab from the repository root before running setup cells.")
else:
    print("[runtime] Runtime diagnostics complete.")


## 4. Local directory contract and state preflight

This preflight cell is intentionally conservative.

It performs safe repairs:

- creates required ignored directories if missing
- creates `.gitkeep` placeholders if missing
- writes a local cleanup helper script under `notebooks/local/`

It fails before the pipeline runs if it detects unsafe or ambiguous local state:

- raw MIT-BIH files without a governed acquisition manifest
- interrupted acquisition staging directories
- hidden `.acquire-*` scratch directories

It does **not** silently delete raw data or generated artifacts.


> **CODE CELL FUNCTION**
>
> Repair safe directory contracts, detect unsafe local state, and write an opt-in cleanup helper.


In [ ]:
# CODE CELL FUNCTION:
# COMMENT MAP:
# - Create only safe ignored directory contracts and placeholder files.
# - Detect manifestless raw files and interrupted acquisition scratch state before pipeline execution.
# - Write a local status payload for diagnostics, but raise on blocked preflight state.
# - Never delete data or artifacts automatically; cleanup remains explicit and opt-in.
# - Stop Run All when the local state cannot support governed artifact generation.
# Repair safe local directory contracts and report unsafe local pipeline state.
#
# This cell is deliberately conservative and strict:
# - It may create missing ignored directories and `.gitkeep` placeholders.
# - It may write a local helper script under `notebooks/local/`.
# - It does not delete raw data, generated artifacts, or experiment outputs.
# - It writes a status file when local state is unsafe, then raises so artifact generation does not proceed.

from __future__ import annotations

import json
import time
from pathlib import Path
from textwrap import dedent
from typing import Any

# The Step 0 status file is the notebook-level contract between cells. It lets
# the preflight, pipeline runner, and readiness checker communicate outcomes
# without relying on exceptions or fragile parsing of displayed output.
STATUS_PATH = Path("notebooks/local/step0-pipeline-status.json")

# These directories are intentionally ignored/rebuildable local working areas.
# Creating the directories is safe; deleting their contents is handled only by the
# explicit opt-in cleanup cell.
REQUIRED_DIRS = [
    Path("data/raw"),
    Path("data/interim"),
    Path("data/processed"),
    Path("data/external"),
    Path("artifacts"),
]

# MIT-BIH raw data locations used by the governed acquisition stage.
RAW_DATA_ROOT = Path("data/raw")
# Dataset-specific subtree beneath RAW_DATA_ROOT.
RAW_MITDB_ROOT = Path("data/raw/mitdb")
# Version-pinned subtree the acquisition manifest below is scoped to.
RAW_VERSION_DIR = Path("data/raw/mitdb/1.0.0")
# Presence of this manifest is what distinguishes a governed acquisition from
# manually copied or interrupted raw files (see find_manifestless_raw_files below).
ACQUISITION_MANIFEST = Path("artifacts/datasets/mitdb/1.0.0/acquisition.json")

# The helper script is written to notebooks/local so it stays local/ignored and
# does not become part of the public source tree.
REMEDIATION_SCRIPT = Path("notebooks/local/clean-local-pipeline-state.fish")


def write_status(status: str, reason: str, message: str, **extra: Any) -> None:
    """Write a small machine-readable Step 0 status payload.

    The notebook uses this file as an explicit cell-to-cell contract. That keeps
    later cells from needing to infer state from exceptions, scrollback, or
    partially displayed CLI output.
    """

    # Ensure the local status directory exists before writing. This path is local
    # runtime state and should not be committed.
    STATUS_PATH.parent.mkdir(parents=True, exist_ok=True)

    # Keep the payload small and explicit so downstream notebooks can inspect it
    # without importing project code.
    payload: dict[str, Any] = {
        "status": status,
        "reason": reason,
        "message": message,
        "written_at_epoch": int(time.time()),
    }
    payload.update(extra)

    # Pretty JSON is useful when users open the status file while debugging.
    STATUS_PATH.write_text(json.dumps(payload, indent=2, sort_keys=True) + "\n")


def repository_root_detected() -> bool:
    """Return True when the kernel appears to be running from repository root."""

    # Both paths are expected in the project root. This keeps the check simple and
    # avoids importing application code before dependency bootstrap.
    return Path("pyproject.toml").exists() and Path("configs").exists()


def ensure_directory_contracts(required_dirs: list[Path]) -> tuple[list[str], list[str]]:
    """Create safe ignored directory contracts and `.gitkeep` placeholders.

    The function is intentionally non-destructive. It only creates missing
    directories/placeholders and returns what it created for transparent logging.
    """

    created_dirs: list[str] = []
    created_gitkeeps: list[str] = []

    # Walk every required directory so one pass creates both the directory itself
    # and its .gitkeep placeholder, keeping the two contracts in sync.
    for directory in required_dirs:
        # Creating an empty ignored directory is safe because it does not remove or
        # overwrite user data. Existing directories are left untouched.
        if not directory.exists():
            directory.mkdir(parents=True, exist_ok=True)
            created_dirs.append(str(directory))

        # `.gitkeep` files document and preserve directory contracts. Creating a
        # missing placeholder is safe even when the directory contains generated
        # files because it does not modify those generated files.
        gitkeep = directory / ".gitkeep"
        # touch() is a no-op if the file already exists elsewhere in this pass, but
        # the exists() check first still lets created_gitkeeps report accurately.
        if not gitkeep.exists():
            gitkeep.touch()
            created_gitkeeps.append(str(gitkeep))

    return created_dirs, created_gitkeeps


def find_manifestless_raw_files() -> list[Path]:
    """Return raw MIT-BIH WFDB files that exist without an acquisition manifest."""

    # No canonical raw directory means there are no manifestless canonical raw
    # files to report.
    if not RAW_VERSION_DIR.exists():
        return []

    # When the manifest exists, the CLI remains responsible for deeper checksum
    # and inventory validation. This notebook only detects the obvious unsafe case.
    if ACQUISITION_MANIFEST.exists():
        return []

    # WFDB records are represented by `.hea`, `.dat`, and `.atr` files. If these
    # exist without the manifest, the acquisition provenance is ambiguous.
    return sorted(
        path
        for path in RAW_VERSION_DIR.iterdir()
        if path.is_file() and path.suffix in {".atr", ".dat", ".hea"}
    )


def find_interrupted_acquisition_state() -> tuple[list[Path], list[Path]]:
    """Find interrupted acquisition staging directories and hidden scratch dirs."""

    # Quarantine/staging directories can remain after interrupted or intentionally
    # stopped acquisition attempts.
    unmanifested_dirs = (
        sorted(RAW_MITDB_ROOT.glob("1.0.0.unmanifested-*"))
        if RAW_MITDB_ROOT.exists()
        else []
    )

    # Hidden `.acquire-*` directories are scratch locations used during protected
    # acquisition flows. A clean checkout should not contain them.
    acquire_scratch_dirs = (
        sorted(path for path in RAW_DATA_ROOT.rglob(".acquire-*") if path.is_dir())
        if RAW_DATA_ROOT.exists()
        else []
    )

    return unmanifested_dirs, acquire_scratch_dirs


def write_cleanup_helper(path: Path) -> None:
    """Write a fish cleanup helper for explicit, user-triggered local rebuilds."""

    # The helper is intentionally explicit and destructive. It exists so the user
    # can inspect the commands before running them. The notebook does not invoke
    # this helper automatically.
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(
        dedent(
            """\
            #!/usr/bin/env fish
            # Clean ignored/rebuildable local pipeline state for ecg_anomaly_detection.
            # Run only from repository root and only when Step 0 tells you local state is unsafe.

            printf '\n## Remove raw MIT-BIH files and interrupted acquisition scratch directories\n'
            rm -rf data/raw/mitdb
            find data/raw -maxdepth 8 -name '.acquire-*' -type d -exec rm -rf {} + 2>/dev/null; or true

            printf '\n## Remove rebuildable generated outputs\n'
            rm -rf data/interim data/processed artifacts

            printf '\n## Recreate required directory contracts\n'
            mkdir -p data/raw data/interim data/processed data/external artifacts notebooks/local
            touch data/raw/.gitkeep data/interim/.gitkeep data/processed/.gitkeep data/external/.gitkeep artifacts/.gitkeep

            printf '\n## Remaining generated/raw files\n'
            find data artifacts -maxdepth 4 -type f 2>/dev/null | sort
            """
        )
    )
    path.chmod(0o755)


# Start with a clear console marker so users can visually separate this cell from
# earlier environment checks.
print("[preflight] Starting safe local-state preflight.")

# Branch the entire preflight on the same repository-root check the diagnostics
# cell above reported, since every remaining check assumes paths relative to root.
if not repository_root_detected():
    # Repository-root detection is a hard prerequisite for artifact generation.
    # Write the status file first, then stop the notebook explicitly.
    write_status(
        status="blocked",
        reason="REPOSITORY_ROOT_NOT_DETECTED",
        message="Repository root was not detected. Start JupyterLab from repository root, then rerun Step 0.",
    )
    print("[preflight] BLOCKED: repository root was not detected.")
    print("[preflight] Start JupyterLab from repository root, then rerun Step 0.")
    raise RuntimeError("Step 0 preflight failed: repository root was not detected.")
else:
    # Safe repairs happen before unsafe-state detection so later cells can rely on
    # the expected local directory skeleton existing.
    created_dirs, created_gitkeeps = ensure_directory_contracts(REQUIRED_DIRS)
    write_cleanup_helper(REMEDIATION_SCRIPT)

    # Gather all unsafe conditions before reporting so the user gets one complete
    # remediation summary instead of a sequence of one-at-a-time failures.
    manifestless_raw_files = find_manifestless_raw_files()
    unmanifested_dirs, acquire_scratch_dirs = find_interrupted_acquisition_state()
    problems: list[str] = []

    # Manifestless raw files are the ambiguous-provenance case: files exist but
    # nothing records where they came from or verifies their integrity.
    if manifestless_raw_files:
        preview = "\n".join(f"  - {path}" for path in manifestless_raw_files[:12])
        # Cap the printed preview so one large stray directory can't flood the cell's
        # output; the full count is still reported alongside the truncated list.
        if len(manifestless_raw_files) > 12:
            preview += f"\n  - ... {len(manifestless_raw_files) - 12} more files"
        problems.append(
            "Raw MIT-BIH files exist without the governed acquisition manifest.\n"
            f"{preview}\n"
            "This usually means manual download/copy or interrupted acquisition."
        )

    # Quarantined "1.0.0.unmanifested-*" staging directories mean a previous
    # acquisition attempt was interrupted before it could validate and promote them.
    if unmanifested_dirs:
        problems.append(
            "Interrupted or quarantined acquisition directories exist:\n"
            + "\n".join(f"  - {path}" for path in unmanifested_dirs)
        )

    # Hidden ".acquire-*" scratch directories are a separate interrupted-state
    # signal from unmanifested_dirs above: mid-download staging rather than a
    # completed-but-unpromoted attempt.
    if acquire_scratch_dirs:
        problems.append(
            "Hidden interrupted acquisition scratch directories exist:\n"
            + "\n".join(f"  - {path}" for path in acquire_scratch_dirs)
        )

    print("[preflight] Safe directory-contract checks complete.")

    # Report only what actually changed; an already-complete directory skeleton
    # produces no output here rather than a misleading "created" list.
    if created_dirs:
        print("[preflight] Created missing directories:")
        # List each created directory individually so the log is a precise, greppable record.
        for item in created_dirs:
            print(f"  - {item}")

    # Same "only report what changed" rule as created_dirs above, for placeholders.
    if created_gitkeeps:
        print("[preflight] Created missing .gitkeep placeholders:")
        # Mirror the created_dirs loop above for the same per-item transparency.
        for item in created_gitkeeps:
            print(f"  - {item}")

    print(f"[preflight] Wrote cleanup helper: {REMEDIATION_SCRIPT}")

    # Every problem gathered above (manifestless files, unmanifested dirs, scratch
    # dirs) converges here: any one of them is enough to block artifact generation.
    if problems:
        remediation = f"fish {REMEDIATION_SCRIPT}"
        write_status(
            status="blocked",
            reason="UNSAFE_LOCAL_PIPELINE_STATE",
            message="Step 0 preflight found unsafe local pipeline state.",
            problems=problems,
            remediation=remediation,
        )
        print("[preflight] BLOCKED: unsafe local pipeline state found.")
        # Number each problem so the printed remediation guidance is easy to
        # cross-reference against the written status file's own problems list.
        for problem_number, problem in enumerate(problems, start=1):
            print(f"[preflight] Problem {problem_number}:")
            print(problem)
        print("[preflight] Recommended cleanup, after reviewing commands:")
        print(f"  {remediation}")
        print("[preflight] The notebook did not delete these files automatically.")
        raise RuntimeError("Step 0 preflight failed: unsafe local pipeline state found. Review the remediation helper before rerunning.")
    else:
        write_status(
            status="ready",
            reason="PREFLIGHT_PASSED",
            message="Safe local-state preflight passed.",
        )
        print("[preflight] No unsafe local acquisition state detected.")
        print("[preflight] Continue to governed artifact generation.")


## 4.1 Optional clean rebuild

<div role="alert" aria-label="Destructive cleanup warning" style="border: 3px solid #b42318; border-left-width: 8px; border-radius: 6px; background: #fff4f2; color: #5c160f; padding: 1rem 1.25rem; margin: 1rem 0; line-height: 1.5;"><strong style="display: block; font-size: 1.15rem; margin-bottom: 0.35rem;">Destructive cleanup — opt in only</strong>This cell is disabled by default. Use it only when the preflight or pipeline diagnostics identify unsafe local ignored state.</div>

It deletes ignored/rebuildable local state, including raw MIT-BIH files, hidden acquisition scratch directories, generated interim files, processed indexes, and artifacts.

It does **not** modify source code, configs, or tracked documentation.


> **CODE CELL FUNCTION**
>
> Optional destructive cleanup of ignored local pipeline state. Disabled by default.


In [ ]:
%%bash
# COMMENT MAP:
# - Validate the local generated-state boundary before invoking the governed pipeline.
# - Treat stale, partial, or inconsistent artifact state as a setup failure requiring correction.
# - Do not allow Step 0 to report success unless downstream model-ready artifacts can exist.
# - Preserve the distinction between local cleanup issues and real pipeline contract failures.
set -Eeuo pipefail

CLEAN_LOCAL_PIPELINE_STATE=0

printf '[cleanup] CLEAN_LOCAL_PIPELINE_STATE=%s.\n' "$CLEAN_LOCAL_PIPELINE_STATE"

if [[ "$CLEAN_LOCAL_PIPELINE_STATE" != "1" ]]; then
  printf '[cleanup] Disabled by default. Set CLEAN_LOCAL_PIPELINE_STATE=1 to remove ignored local pipeline outputs.\n'
  printf '[cleanup] This cell would run these operations from repository root:\n'
  printf '[cleanup]   rm -rf data/raw/mitdb\n'
  printf "[cleanup]   find data/raw -maxdepth 8 -name '.acquire-*' -type d -exec rm -rf {} +\n"
  printf '[cleanup]   rm -rf data/interim data/processed artifacts\n'
  printf '[cleanup]   mkdir -p data/raw data/interim data/processed data/external artifacts notebooks/local\n'
  printf '[cleanup]   touch data/raw/.gitkeep data/interim/.gitkeep data/processed/.gitkeep data/external/.gitkeep artifacts/.gitkeep\n'
  exit 0
fi

test -f pyproject.toml || {
  printf '[cleanup] ERROR: pyproject.toml not found. Run this notebook from repository root.\n' >&2
  exit 1
}

printf '[cleanup] Removing raw MIT-BIH acquisition state, including hidden scratch dirs...\n'
rm -rf data/raw/mitdb
find data/raw -maxdepth 8 -name '.acquire-*' -type d -exec rm -rf {} + 2>/dev/null || true

printf '[cleanup] Removing rebuildable generated pipeline outputs...\n'
rm -rf data/interim data/processed artifacts

printf '[cleanup] Restoring local directory contracts and placeholders...\n'
mkdir -p data/raw data/interim data/processed data/external artifacts notebooks/local
touch data/raw/.gitkeep data/interim/.gitkeep data/processed/.gitkeep data/external/.gitkeep artifacts/.gitkeep

printf '[cleanup] Cleanup complete. Rerun the preflight cell before running the pipeline.\n'
find data artifacts -maxdepth 4 -type f 2>/dev/null | sort


## 5. Generate governed local artifacts

This cell runs the governed pipeline command. It reports periodic progress and captures the full CLI output to a temporary log file.

Important behavior:

- retries are disabled by default
- cleanup is never performed silently
- known local-state failures are reported with recovery guidance
- confirmed channel-identity mismatch is reported as **PIPE-006**, not as another cleanup request

The pipeline command is intentionally not reimplemented in this notebook; see the [pipeline orchestration guide](../docs/pipeline-orchestration.md) for the maintained command and stage contract.


> **CODE CELL FUNCTION**
>
> Run the governed local pipeline with progress output and failure classification. Known, classified failures write `notebooks/local/step0-pipeline-status.json` and do **not** dump a raw `CalledProcessError` wall.


In [ ]:
# CODE CELL FUNCTION:
# Run the governed pipeline as a strict prerequisite for downstream public notebooks.
#
# COMMENT MAP:
# - Execute the supported package CLI; do not reimplement pipeline stages in the notebook.
# - Stream output so a public reviewer can see progress during acquisition/artifact generation.
# - Write a machine-readable status file for diagnostics and downstream inspection.
# - Treat any nonzero pipeline exit as a real failure, not a successful notebook outcome.
# - Raise immediately after classification so Run All stops before Step 1/Step 2 dependencies are faked.
# - Keep PIPE-006 diagnostics explicit because it blocks model-ready artifact generation.

from __future__ import annotations

import json
import subprocess
import tempfile
import time
from collections import Counter
from pathlib import Path
from typing import Any

# Shared cell-to-cell contract file, matching the preflight cell's own STATUS_PATH.
STATUS_PATH = Path("notebooks/local/step0-pipeline-status.json")
# Referenced in the PIPE-006 status payload and printed diagnostics below so a
# reviewer hitting this specific failure has one canonical place to read more.
PIPE_006_URL = "https://github.com/Jared-Godar/ecg_anomaly_detection/issues/56"

# This is the supported CLI command. The notebook owns public UX around the call;
# the package owns acquisition, validation, windowing, splitting, training, and evaluation.
PIPELINE_COMMAND = [
    "uv",
    "run",
    "ecg-data",
    "run-pipeline",
    "--repository-root",
    ".",
    "--dataset-config",
    "configs/mitdb-v1.0.0.toml",
    "--mapping-config",
    "configs/annotation-map-v1.toml",
    "--window-config",
    "configs/windowing-v1.toml",
    "--split-config",
    "configs/splitting-v2.toml",
    "--training-config",
    "configs/training-baseline-v1.toml",
    "--evaluation-config",
    "configs/evaluation-baseline-v1.toml",
]


def write_status(status: str, reason: str, message: str, **extra: Any) -> None:
    """Persist Step 0 status for diagnostics without pretending failure is success."""
    # Downstream notebooks may inspect this file, but a failed status is not a pass.
    # The notebook still raises after writing the file when artifact generation fails.
    STATUS_PATH.parent.mkdir(parents=True, exist_ok=True)
    payload: dict[str, Any] = {
        "status": status,
        "reason": reason,
        "message": message,
        "issue": PIPE_006_URL if reason == "PIPE-006" else None,
        "written_at_epoch": int(time.time()),
    }
    payload.update(extra)
    STATUS_PATH.write_text(json.dumps(payload, indent=2, sort_keys=True) + "\n", encoding="utf-8")


def load_status() -> dict[str, Any] | None:
    """Load an existing Step 0 status file when preflight already marked the run blocked."""
    # No status file means preflight has not run (or already cleared its own state);
    # either way there is nothing here to block on.
    if not STATUS_PATH.exists():
        return None
    # A status file that fails to parse is treated as its own failure state rather
    # than propagating a raw JSONDecodeError out of this helper.
    try:
        return json.loads(STATUS_PATH.read_text(encoding="utf-8"))
    except json.JSONDecodeError as exc:
        return {
            "status": "failed",
            "reason": "STATUS_FILE_INVALID_JSON",
            "message": f"Could not parse {STATUS_PATH}: {exc}",
        }


def latest_run_directory() -> Path | None:
    """Return the most recently modified generated run directory, if one exists."""
    runs_root = Path("artifacts/runs")
    # No runs directory at all means no pipeline run has ever produced output yet.
    if not runs_root.exists():
        return None
    run_dirs = [path for path in runs_root.iterdir() if path.is_dir()]
    # A runs directory can exist but be empty right after `mkdir -p`; treat that the
    # same as "no runs directory" rather than crashing on an empty max().
    if not run_dirs:
        return None
    return max(run_dirs, key=lambda path: path.stat().st_mtime)


def summarize_window_channels() -> dict[str, Any]:
    """Summarize selected channel identities for the latest generated run."""
    run_dir = latest_run_directory()
    # No run yet means there is nothing to summarize; the caller (PIPE-006 handling)
    # treats an empty summary as "no diagnostic detail available", not an error.
    if run_dir is None:
        return {"run_dir": None, "counts": {}, "non_mlii_records": [], "records": []}

    records: dict[str, str] = {}
    # Use one row per record in the latest run so repeated failed attempts do not
    # inflate counts. PIPE-006 is about identity consistency, not retry volume.
    for metadata_path in sorted((run_dir / "windows").glob("*.json")):
        record_id = metadata_path.stem
        # A per-record metadata file should always be valid JSON; treat corruption
        # as a diagnosable data point instead of letting it abort the whole summary.
        try:
            metadata = json.loads(metadata_path.read_text(encoding="utf-8"))
        except json.JSONDecodeError:
            records[record_id] = "<invalid-json>"
            continue
        records[record_id] = str(metadata.get("channel_name", "<missing>"))

    counts = Counter(records.values())
    non_mlii_records = [
        {"record_id": record_id, "channel_name": channel_name}
        for record_id, channel_name in sorted(records.items())
        if channel_name != "MLII"
    ]
    return {
        "run_dir": str(run_dir),
        "counts": dict(sorted(counts.items())),
        "non_mlii_records": non_mlii_records,
        "records": [
            {"record_id": record_id, "channel_name": channel_name}
            for record_id, channel_name in sorted(records.items())
        ],
    }


def print_channel_summary(summary: dict[str, Any]) -> None:
    """Print PIPE-006 channel diagnostics in a compact human-readable form."""
    counts = summary.get("counts", {}) or {}
    non_mlii_records = summary.get("non_mlii_records", []) or []

    # An empty summary (see summarize_window_channels' run_dir is None case) prints
    # nothing here rather than a misleading "0 records" section.
    if counts:
        print("[pipeline] Observed selected channel identity counts:")
        # Break down by channel identity so a reviewer can see the full distribution,
        # not just whether a mismatch exists.
        for channel_name, count in sorted(counts.items()):
            print(f"  - {channel_name}: {count} records")
    # This is the actionable half of the diagnostic: which specific records disagree
    # with the expected MLII channel, so they can be inspected individually.
    if non_mlii_records:
        print("[pipeline] Records whose selected channel is not MLII:")
        # List each mismatched record by ID so it can be looked up directly in the
        # run's windows/ directory.
        for row in non_mlii_records:
            print(f"  - {row.get('record_id')}: {row.get('channel_name')}")


def classify_pipeline_failure(output_text: str) -> tuple[str, str, dict[str, Any]]:
    """Classify a failed pipeline run without converting it into notebook success."""
    lower_output = output_text.lower()
    # Each check below is independent (not elif) and ordered by specificity: the
    # first substring match wins, so a more specific known failure signature must
    # come before the generic PIPELINE_FAILED fallback at the end.
    if "window shards do not share one geometry and configuration identity" in lower_output:
        channel_summary = summarize_window_channels()
        return (
            "PIPE-006",
            "Window extraction produced mixed selected channel identities; model-ready artifacts were not generated.",
            {"channel_summary": channel_summary},
        )
    # Mirrors the preflight cell's own manifestless-file detection, but classifies a
    # pipeline-time failure caused by the same underlying condition.
    if "required files already exist without an acquisition manifest" in lower_output:
        return "MANIFESTLESS_RAW_FILES", "Raw files exist without a governed acquisition manifest.", {}
    # Mirrors the preflight cell's interrupted-scratch-directory detection.
    if "unexpected source file or directory" in lower_output and ".acquire-" in lower_output:
        return "INTERRUPTED_ACQUISITION_SCRATCH", "Interrupted acquisition scratch directories remain under data/raw.", {}
    # A missing output directory usually means required_dirs (preflight cell) was
    # never run, or its contents were deleted between preflight and pipeline execution.
    if "pipeline output base must be an existing regular directory" in lower_output:
        return "MISSING_OUTPUT_DIRECTORY", "A required generated-output directory is missing.", {}
    # No known signature matched; report a generic failure rather than misclassifying it.
    return "PIPELINE_FAILED", "The governed pipeline failed. Review the captured log.", {}


def stream_pipeline(command: list[str], log_path: Path) -> tuple[int, str]:
    """Run the pipeline CLI, stream combined output, and return exit code plus output text."""
    started_at = time.time()
    output_chunks: list[str] = []

    print(f"[pipeline] Starting governed local pipeline at {time.ctime(started_at)}")
    print("[pipeline] This notebook is strict: nonzero pipeline exit stops Step 0.")
    print(f"[pipeline] Full CLI output will be captured at {log_path}")

    # Keep the log file open for the whole subprocess lifetime so every streamed
    # line is written immediately, not buffered and lost on an abnormal exit.
    with log_path.open("w", encoding="utf-8") as log_file:
        process = subprocess.Popen(
            command,
            stdout=subprocess.PIPE,
            stderr=subprocess.STDOUT,
            text=True,
            bufsize=1,
        )
        # Popen only omits stdout when it wasn't requested; PIPE was passed above,
        # so this guards type-checking rather than a realistic runtime path.
        if process.stdout is None:
            return 1, "Pipeline stdout pipe was not created."
        # A raised KeyboardInterrupt here must still clean up the child process
        # (see the except clause below) rather than leaving it running detached.
        try:
            # Stream line-by-line so a public reviewer running this notebook sees
            # live progress instead of a long silent wait for the whole pipeline.
            for line in process.stdout:
                print(line, end="")
                log_file.write(line)
                log_file.flush()
                output_chunks.append(line)
            exit_code = process.wait()
        except KeyboardInterrupt:
            # Interrupts must stop the child process so partial outputs are not left writing in the background.
            print("[pipeline] Interrupt detected. Terminating child pipeline process...")
            process.terminate()
            # Give the process a bounded grace period to exit cleanly after
            # terminate() before escalating to kill(), which cannot be ignored.
            try:
                process.wait(timeout=10)
            except subprocess.TimeoutExpired:
                process.kill()
                process.wait()
            return 130, "".join(output_chunks) + "\nKeyboardInterrupt\n"

    elapsed_seconds = int(time.time() - started_at)
    print(f"[pipeline] Pipeline process ended after {elapsed_seconds // 60:02d}m {elapsed_seconds % 60:02d}s with exit code {exit_code}.")
    print(f"[pipeline] Review full CLI output at: {log_path}")
    return int(exit_code), "".join(output_chunks)


def require_model_ready_artifacts() -> dict[str, str]:
    """Return the generated artifact paths required by Step 1 and Step 2, or raise."""
    # Step 2 cannot train from status text. It needs these generated files.
    processed_runs_dir = Path("data/processed/runs")
    candidates = sorted(processed_runs_dir.glob("*/dataset-index.json"), key=lambda path: path.stat().st_mtime)
    # No dataset index anywhere means the pipeline never reached the indexing
    # stage, regardless of what exit code it reported.
    if not candidates:
        raise RuntimeError("Step 0 did not produce data/processed/runs/<run-id>/dataset-index.json.")

    dataset_index = candidates[-1]
    run_id = dataset_index.parent.name
    run_root = Path("artifacts/runs") / run_id
    required = {
        "dataset_index": dataset_index,
        "split_manifest": run_root / "split.json",
        "split_quality": run_root / "split_quality_summary.json",
        "run_manifest": run_root / "run-manifest.json",
    }
    missing = [f"{label}: {path}" for label, path in required.items() if not path.is_file()]
    # A zero pipeline exit code alone is not sufficient proof of success; every
    # required file must actually exist before Step 1/Step 2 can depend on it.
    if missing:
        raise RuntimeError("Step 0 pipeline completed but required downstream artifacts are missing:\n" + "\n".join(missing))
    return {label: str(path) for label, path in required.items()}


# Re-check the preflight cell's own verdict before spending time on a real pipeline
# run, in case this cell is re-run alone without re-running the preflight cell first.
preflight_status = load_status()
# A None or non-blocked status means it's safe to proceed to the pipeline run below.
if preflight_status and preflight_status.get("status") == "blocked":
    # A blocked preflight is a hard stop because downstream notebooks cannot depend on artifact generation.
    raise RuntimeError(
        "Step 0 preflight is blocked before pipeline execution. Resolve the preflight state before continuing.\n"
        f"Reason: {preflight_status.get('reason')}\n"
        f"Message: {preflight_status.get('message')}"
    )

# A leftover status file from a prior run (pass or fail) must not survive into this
# one's own verdict below.
if STATUS_PATH.exists():
    # Remove stale status so this run cannot inherit a previous pass/fail state.
    STATUS_PATH.unlink()

# A dedicated temp file (rather than STATUS_PATH) holds the full streamed log, since
# the status file itself only records a short summary plus this log's own path.
log_file = Path(tempfile.mkstemp(prefix="ecg-pipeline-notebook.", suffix=".log")[1])
# This is the actual governed pipeline invocation; everything above it is setup and
# state checks, and everything below it interprets this call's result.
exit_code, output_text = stream_pipeline(PIPELINE_COMMAND, log_file)

# A nonzero exit is a real failure; classify it for a specific, actionable message
# instead of a generic traceback, then raise so Run All stops here.
if exit_code != 0:
    reason, message, extra = classify_pipeline_failure(output_text)
    write_status(
        status="failed",
        reason=reason,
        message=message,
        log_file=str(log_file),
        exit_code=exit_code,
        **extra,
    )
    print(f"[pipeline] Classified failure: {reason}")
    print(f"[pipeline] {message}")
    # PIPE-006 gets extra diagnostic detail (per-record channel identities) beyond
    # the generic failure message, since it's a data-integrity issue worth showing.
    if reason == "PIPE-006":
        print_channel_summary(extra.get("channel_summary", {}))
        print("[pipeline] This is a hard blocker for Step 2, not a successful Step 0 run.")
        print(f"[pipeline] See: {PIPE_006_URL}")
    raise RuntimeError(
        "Step 0 governed pipeline failed and did not produce model-ready artifacts. "
        f"Reason: {reason}. Log file: {log_file}"
    )

# A zero exit code alone isn't proof of success (see require_model_ready_artifacts'
# own docstring); this call is what actually verifies the required files exist.
artifact_paths = require_model_ready_artifacts()
write_status(
    status="complete",
    reason="PIPELINE_SUCCEEDED_AND_ARTIFACTS_VERIFIED",
    message="Governed local pipeline completed and required Step 1/Step 2 artifacts are present.",
    log_file=str(log_file),
    artifacts=artifact_paths,
)

print("[pipeline] Step 0 pipeline generation completed successfully.")
print("[pipeline] Required downstream artifacts verified:")
# List each verified artifact by its logical label so Step 1/Step 2 authors can see
# exactly which generated files this run's success depends on.
for label, path in artifact_paths.items():
    print(f"  - {label}: {path}")


## 5.1 Optional post-failure channel diagnostics

Run this only if the pipeline cell fails with `window shards do not share one geometry and configuration identity`.

This diagnostic cell does not modify files. It summarizes selected channel identity by generated window metadata and identifies non-`MLII` records. If this shows `102`, `104`, and `114` resolving to `V5`, that is the PIPE-006 data-contract issue.


> **CODE CELL FUNCTION**
>
> Summarize generated window channel identity metadata without changing local files.


In [ ]:
# CODE CELL FUNCTION:
# COMMENT MAP:
# - Provide post-failure channel diagnostics for PIPE-006 investigations.
# - Read only generated metadata from the latest run directory.
# - Do not mutate data, artifacts, configs, or status files.
# - This cell is only reached during manual inspection after strict Step 0 failure.
# Summarize generated window channel identities after a pipeline failure.
#
# This cell is optional and non-throwing. It does not change files. It is useful
# when Step 0 reports PIPE-006 because it shows exactly which selected signal
# identities appeared in the latest generated run.

from __future__ import annotations

import json
from collections import Counter
from pathlib import Path


def latest_run_directory() -> Path | None:
    """Return the newest generated pipeline run directory, if one exists."""

    # The pipeline writes one UUID-named directory per run under artifacts/runs.
    runs_root = Path("artifacts/runs")
    # If no runs exist, there is nothing diagnostic to summarize.
    if not runs_root.exists():
        return None

    # Keep only directories so placeholder files or editor artifacts do not affect
    # the newest-run selection.
    run_dirs = [path for path in runs_root.iterdir() if path.is_dir()]
    # An existing-but-empty runs_root is treated the same as a missing one.
    if not run_dirs:
        return None

    # Modification time is the best local proxy for the run produced by the most
    # recent pipeline attempt.
    return max(run_dirs, key=lambda path: path.stat().st_mtime)


def load_window_channel_rows(run_dir: Path) -> list[tuple[str, str]]:
    """Load `(record_id, channel_name)` rows from one generated run directory."""

    rows: list[tuple[str, str]] = []
    windows_dir = run_dir / "windows"

    # One metadata JSON file exists per extracted record; iterate all of them so the
    # diagnostic covers every record in the run, not just a sample.
    for metadata_path in sorted(windows_dir.glob("*.json")):
        # `path.stem` is the record ID because metadata files are named `100.json`,
        # `101.json`, etc.
        record_id = metadata_path.stem

        # A metadata file should always be valid JSON; corruption is surfaced as a
        # diagnostic value rather than aborting the whole summary.
        try:
            metadata = json.loads(metadata_path.read_text())
        except json.JSONDecodeError:
            # Preserve the record in the diagnostic output and make the metadata
            # problem visible without failing the notebook cell.
            rows.append((record_id, "<invalid-json>"))
            continue

        # `channel_name` is the resolved physical signal identity. This is the
        # value that exposed PIPE-006 when channel_index=0 selected MLII for most
        # records but V5 for records 102, 104, and 114.
        channel_name = str(metadata.get("channel_name", "<missing>"))
        rows.append((record_id, channel_name))

    return rows


# Select only the latest run so repeated failed attempts do not inflate counts.
run_dir = latest_run_directory()

# No run directory means Step 0 never got far enough to extract windows; there is
# nothing further to diagnose here.
if run_dir is None:
    print("[diagnostics] No generated run directory found under artifacts/runs/.")
else:
    rows = load_window_channel_rows(run_dir)
    counts = Counter(channel_name for _, channel_name in rows)
    non_mlii = [(record_id, channel_name) for record_id, channel_name in rows if channel_name != "MLII"]

    print(f"[diagnostics] Latest generated run: {run_dir}")
    print("[diagnostics] Observed selected channel identities:")
    # Break down by channel identity so a reviewer can see the full distribution at a glance.
    for name, count in sorted(counts.items()):
        print(f"  - {name}: {count} records")

    # A non-empty non_mlii list is exactly the PIPE-006 symptom: mixed channel
    # identities across records in the same run.
    if non_mlii:
        print("[diagnostics] Records whose selected channel is not MLII:")
        # List each mismatched record individually so it can be cross-referenced
        # against the windows/ metadata directly.
        for record_id, channel_name in non_mlii:
            print(f"  - {record_id}: {channel_name}")
        print("[diagnostics] This matches PIPE-006 if records 102, 104, and 114 resolve to V5.")
    else:
        print("[diagnostics] All generated records selected MLII.")


## 6. Verify Step 0 completion

Run this after the governed pipeline completes. It performs a small readiness suite for downstream notebooks.

If PIPE-006 is still open and the pipeline stopped with mixed channel identity, this check should fail clearly and direct you to the pipeline/config issue rather than suggesting another cleanup loop.


> **CODE CELL FUNCTION**
>
> Verify generated artifacts are sufficiently complete for downstream notebooks, without opening protected-test data.


In [ ]:
# CODE CELL FUNCTION:
# Verify Step 0 completion as a hard downstream contract.
#
# COMMENT MAP:
# - Treat Step 0 as successful only when the pipeline status is complete and artifacts exist.
# - Raise on blocked/failed status so downstream notebooks are not tested against fake readiness.
# - Verify the exact model-ready artifacts Step 1 and Step 2 rely on.
# - Preserve protected-test discipline by checking metadata paths only, not opening test data.

from __future__ import annotations

import json
from pathlib import Path
from typing import Any

# Matches the preflight and pipeline-execution cells' own STATUS_PATH so all three
# cells read and write the same cell-to-cell contract file.
STATUS_PATH = Path("notebooks/local/step0-pipeline-status.json")


def load_step0_status() -> dict[str, Any]:
    """Load the status written by strict Step 0 pipeline execution."""
    # A missing status file means the pipeline-execution cell never ran; this is a
    # hard downstream-contract violation, not a soft warning.
    if not STATUS_PATH.is_file():
        raise RuntimeError(
            "Step 0 status file was not written. Run the governed pipeline cell before continuing."
        )
    # A status file that fails to parse means it was corrupted or hand-edited; treat
    # that as a contract violation rather than propagating a raw JSONDecodeError.
    try:
        status = json.loads(STATUS_PATH.read_text(encoding="utf-8"))
    except json.JSONDecodeError as exc:
        raise RuntimeError(f"Step 0 status file is invalid JSON: {STATUS_PATH}") from exc
    # json.loads can return any JSON type; only an object matches this contract's
    # expected shape (status/reason/message/... keys read below and by callers).
    if not isinstance(status, dict):
        raise RuntimeError(f"Step 0 status file must contain a JSON object: {STATUS_PATH}")
    return status


def latest_dataset_index() -> Path:
    """Return the newest generated dataset index or raise a downstream-contract error."""
    candidates = sorted(
        Path("data/processed/runs").glob("*/dataset-index.json"),
        key=lambda path: path.stat().st_mtime,
    )
    # No dataset index anywhere means Step 0 never reached the indexing stage,
    # regardless of what its own status file claims.
    if not candidates:
        raise RuntimeError("Missing required Step 0 output: data/processed/runs/<run-id>/dataset-index.json")
    return candidates[-1]


# Load Step 0's own verdict first; every check below assumes it succeeded.
status = load_step0_status()
# A blocked/failed status must raise immediately rather than proceeding to the
# artifact-existence checks below, which assume a completed run.
if status.get("status") != "complete":
    raise RuntimeError(
        "Step 0 did not complete successfully. Downstream notebooks must not run.\n"
        f"Status: {status.get('status')}\n"
        f"Reason: {status.get('reason')}\n"
        f"Message: {status.get('message')}\n"
        f"Log file: {status.get('log_file')}"
    )

# A "complete" status alone is not sufficient proof; re-derive and re-verify the
# actual generated artifacts independently of what the status file claims.
index_path = latest_dataset_index()
# The dataset index's own parent directory name is the run ID that produced it.
run_id = index_path.parent.name
# The remaining required artifacts live in this run's artifacts/runs/<run-id>/ tree.
run_root = Path("artifacts/runs") / run_id
# The exact set of generated files Step 1 and Step 2 depend on, labeled for the
# readable missing-artifact report below.
required_artifacts = {
    "dataset_index": index_path,
    "split_manifest": run_root / "split.json",
    "split_quality": run_root / "split_quality_summary.json",
    "run_manifest": run_root / "run-manifest.json",
}
# Verify every required file actually exists on disk, not just that the status file
# reported success.
missing = [f"{label}: {path}" for label, path in required_artifacts.items() if not path.is_file()]
# Any missing artifact makes downstream notebooks unsafe to run, regardless of how
# many of the required files are actually present.
if missing:
    raise RuntimeError("Step 0 status says complete, but required downstream artifacts are missing:\n" + "\n".join(missing))

{
    "step_0_status": status.get("status"),
    "reason": status.get("reason"),
    "run_id": run_id,
    "artifacts_verified": {label: str(path) for label, path in required_artifacts.items()},
    "next_notebook": "notebooks/01-narrative-walkthrough.ipynb",
    "test_partition_opened": False,
}


## 7. Continue to Step 1

After this notebook completes successfully, continue with the [`01` narrative walkthrough](./01-narrative-walkthrough.ipynb), then run the [`02` gradient boosting validation example](./02-high-performing-gradient-boosting-validation.ipynb).

<div role="alert" aria-label="Blocked pipeline state" style="border: 3px solid #b42318; border-left-width: 8px; border-radius: 6px; background: #fff4f2; color: #5c160f; padding: 1rem 1.25rem; margin: 1rem 0; line-height: 1.5;"><strong style="display: block; font-size: 1.15rem; margin-bottom: 0.35rem;">Blocked state: resolve PIPE-006 first</strong>If Step 0 stops on PIPE-006, do not continue to downstream execution. The notebook has identified a real pipeline/channel-contract issue that must be resolved separately.</div>
